In [ ]:
!pip install chromadb sentence-transformers groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [ ]:
import os
import chromadb
from groq import Groq
from google.colab import userdata

# **SETUP**

In [ ]:
chroma_client=chromadb.PersistentClient(path='./chroma_db')
collection=chroma_client.get_or_create_collection(name='ABC_Handbook')
groq_client=Groq(api_key=userdata.get('GROQ_API'))

In [ ]:
# --- Document: ABC Employee Handbook (chunked by section) ---
handbook_chunks = [
    "### Company Structure\nABC was founded in 2024 by Mark and John. The company operates in the AI education space with teams across engineering, content, and operations.",
    "### Work Hours\nStandard work hours are 9 AM to 6 PM, Monday through Friday. Flexible start times between 8–10 AM are permitted with manager approval.",
    "### Leave Policy\nEmployees are entitled to 12 casual leaves and 6 sick leaves per year. Leaves do not carry forward to the next year.",
    "### Remote Work\nEmployees must be present in the office on Tuesdays and Wednesdays. Remote work is permitted on all other working days.",
    "### Onboarding\nNew employees are expected to complete the onboarding module within two weeks of joining. The module covers company tools, processes, and code of conduct.",
    "### Learning and Development\nABC provides access to learning platforms. Priority courses are: Python for Data Science, SQL Fundamentals, and Machine Learning Foundations.",
]


In [ ]:
collection.upsert(
  documents=handbook_chunks,
  ids=[f'chunk_{i}' for i in range(len(handbook_chunks))]
)
print(f'Indexed {len(handbook_chunks)} Chunks')

Indexed 6 Chunks


# **RAG QUERY FUNCTION**

In [ ]:
def ask(question:str, top_k:int=3) -> str:
  #Retrive Relevent chunks
  results=collection.query(
    query_texts=[question],
    n_resilts=top_k
  )
  retrieved=results["documents"][0]
  context = "\n\n".join(retrieved)

# **BUILD THE RAG PROMPT**

In [ ]:
def ask(question:str, top_k:int=3) -> str:
  #Retrive Relevent chunks
  results=collection.query(
    query_texts=[question],
    n_results=top_k
  )
  retrieved=results["documents"][0]
  context = "\n\n".join(retrieved)
  prompt = f"""Answer the user question using only the context provided below.
If the answer cannot be found in the context, reply exactly: "I am not sure."
Do not use outside knowledge.

  context : {context}

    Question: {question}"""

  response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
  )
  return response.choices[0].message.content

In [ ]:
questions=[
    "How many casual leaves and sick leaves do employees get ?",
    "Who Founded ABC",
    "Who is the founder of CVR College?"
]
for q in questions:
  print(f"Question: {q}")
  print(f"Answer: {ask(q)}\n")


Question: How many casual leaves and sick leaves do employees get ?
Answer: Employees are entitled to 12 casual leaves and 6 sick leaves per year.

Question: Who Founded ABC
Answer: Mark and John.

Question: Who is the founder of CVR College?
Answer: I am not sure.

